# Đo lại pass@1 HumanEval / HumanEval+: v4 (merged), base và Instruct

EvalPlus + vLLM, greedy, 1 mẫu mỗi bài. Mỗi lần chạy là một cặp (model, kiểu prompt):

| Kiểu prompt | Cờ EvalPlus | Prompt gửi cho model |
|---|---|---|
| `base` | `--force-base-prompt` | completion thuần: signature + docstring, model viết tiếp |
| `chatml` | (không cờ) | chat template của tokenizer (ChatML, kèm system prompt mặc định của Qwen) |

Mặc định 5 lần chạy: `v4:base`, `v4:chatml`, `base:base`, `base:chatml`, `instruct:chatml` (Instruct không chạy kiểu base vì đó không phải cách dùng chuẩn của nó).
Cuối notebook in bảng HE/HE+ kèm khoảng tin cậy 95% và so sánh ghép cặp (McNemar chính xác) với lần chạy mốc `REF`.

## 1. Cài đặt

In [1]:
!pip install -q uv
!uv venv /kaggle/working/vllm_env --python 3.12
!uv pip install -q --python /kaggle/working/vllm_env/bin/python "evalplus[vllm]"
!/kaggle/working/vllm_env/bin/python -c "import torch, vllm; print('torch', torch.__version__, 'cuda', torch.version.cuda, '| vllm', vllm.__version__)"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 92.6 MB/s eta 0:00:00
Using CPython 3.12.3 interpreter at: /usr/bin/python3.12
Creating virtual environment at: vllm_env
Activate with: source vllm_env/bin/activate
torch 2.13.0+cu130 cuda 13.0 | vllm 0.32.0


## 2. Config

In [2]:
import os
from datetime import datetime
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

MODELS = {
    "v4":       "hoangnh39/qwen2.5-coder-7b-opencode-v4-merged",
    "base":     "Qwen/Qwen2.5-Coder-7B",
    "instruct": "Qwen/Qwen2.5-Coder-7B-Instruct",
}
RUNS = ["v4:base", "v4:chatml", "base:base", "instruct:chatml"]  
REF = "instruct_chatml"  
TP, DTYPE = 2, "float16"
EVAL = "/kaggle/working/vllm_env/bin/evalplus.evaluate"

RESULTS = "evalplus_results/he_" + datetime.now().strftime("%Y%m%d_%H%M")
os.makedirs(RESULTS, exist_ok=True)
print("Kết quả:", RESULTS)

!nvidia-smi --query-gpu=name,memory.total --format=csv

Kết quả: evalplus_results/he_20261102_0728
name, memory.total [MiB]
Tesla T4, 15360 MiB
Tesla T4, 15360 MiB


## 3. Kiểm tra trước khi chạy

EvalPlus chỉ dùng chat template khi tokenizer của model có `chat_template`; thiếu thì kiểu `chatml` âm thầm chạy giống `base`. Cell này dừng sớm nếu thiếu, và in token dừng của model.

In [3]:
from transformers import AutoTokenizer, GenerationConfig

for tag in sorted({r.split(":")[0] for r in RUNS if r.endswith(":chatml")}):
    model = MODELS[tag]
    tok = AutoTokenizer.from_pretrained(model)
    assert tok.chat_template, f"{model}: tokenizer thiếu chat_template -> lần chạy chatml sẽ chạy như base"
    print(f"{tag}: chat_template OK | generation_config eos = {GenerationConfig.from_pretrained(model).eos_token_id}")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

... (4 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/2.51k [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/139 [00:00<?, ?B/s]

v4: chat_template OK | generation_config eos = [151645, 151643]


## 4. Chạy benchmark

Mỗi lần chạy là một tiến trình riêng nên GPU được giải phóng giữa các lần. Log: `evalplus_<tag>_<prompt>_<dataset>.log`; mẫu và kết quả chi tiết: `evalplus_results/<prompt>/<dataset>/`.

In [4]:
import glob

FLAGS = {"base": "--force-base-prompt", "chatml": ""}
for run in RUNS:
    tag, prompt = run.split(":")
    name = f"{tag}_{prompt}"
    if glob.glob(f"{RESULTS}/{name}/humaneval/*_eval_results.json"):
        print(f"===== {name}: đã có kết quả, bỏ qua =====")
        continue
    model, flag = MODELS[tag], FLAGS[prompt]
    print(f"===== {name}: {model} =====", flush=True)
    !{EVAL} --model {model} --dataset humaneval --backend vllm --greedy {flag} --root {RESULTS}/{name} --tp {TP} --dtype {DTYPE} 2>&1 | tee {RESULTS}/{name}.log | tail -n 12

===== v4_base: hoangnh39/qwen2.5-coder-7b-opencode-v4-merged =====
(EngineCore pid=220) INFO 11-02 07:41:53 [multiproc_executor.py:479] [shutdown] Executor: waiting for worker exit count=2
(Worker_TP0 pid=228) INFO 11-02 07:41:53 [multiproc_executor.py:843] Parent process exited, terminating worker queues
(EngineCore pid=220) INFO 11-02 07:41:55 [multiproc_executor.py:486] [shutdown] Executor: all workers exited gracefully
Computing expected output...
Expected outputs computed in 24.98s
Reading samples...
164it [00:00, 2042.43it/s]
100%|██████████| 164/164 [00:19<00:00,  8.24it/s]
humaneval (base tests)
pass@1:	0.750
humaneval+ (base + extra tests)
pass@1:	0.683
===== v4_chatml: hoangnh39/qwen2.5-coder-7b-opencode-v4-merged =====
(EngineCore pid=1036) INFO 11-02 08:12:16 [core.py:1499] [shutdown] EngineCore: exiting busy loop
(EngineCore pid=1036) INFO 11-02 08:12:16 [multiproc_executor.py:479] [shutdown] Executor: waiting for worker exit count=2
(Worker_TP0 pid=1044) INFO 11-02 08:12:

## 5. Summary

In [5]:
import csv, json, math
from pathlib import Path

def load(path):
    """{task_id: (đúng test gốc, đúng cả test gốc và +)} từ eval_results.json của EvalPlus (1 mẫu mỗi bài)."""
    ev = json.loads(Path(path).read_text(encoding="utf-8"))["eval"]
    out = {}
    for task, samples in ev.items():
        s = samples[0]
        base = s["base_status"] == "pass"
        out[task] = (base, base and s["plus_status"] == "pass")
    return out

def wilson(k, n, z=1.96):
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return 100 * (c - h), 100 * (c + h)

def mcnemar(a, b):
    """(chỉ a đúng, chỉ b đúng, p hai phía chính xác) cho hai dict {task: bool}."""
    only_a = sum(1 for t in a if a[t] and not b[t])
    only_b = sum(1 for t in a if b[t] and not a[t])
    n = only_a + only_b
    if n == 0:
        return only_a, only_b, 1.0
    p = 2 * sum(math.comb(n, i) for i in range(min(only_a, only_b) + 1)) / 2 ** n
    return only_a, only_b, min(1.0, p)

runs = {}
for d in sorted(p for p in Path(RESULTS).iterdir() if p.is_dir()):
    files = sorted(d.glob("humaneval/*_eval_results.json"))
    if files:
        runs[d.name] = load(files[-1])
    else:
        print(f"(bỏ qua {d.name}: chưa có eval_results.json)")
assert runs, f"Không thấy kết quả nào trong {RESULTS}"

print(f"{'lần chạy':<22}{'HE':>14}{'HE+':>14}   95% CI của HE+")
rows = []
for name, r in runs.items():
    n = len(r)
    b = sum(v[0] for v in r.values())
    p = sum(v[1] for v in r.values())
    lo, hi = wilson(p, n)
    print(f"{name:<22}{f'{b}/{n} {100 * b / n:.1f}':>14}{f'{p}/{n} {100 * p / n:.1f}':>14}   {lo:.1f} – {hi:.1f}")
    rows.append([name, n, b, round(100 * b / n, 1), p, round(100 * p / n, 1)])

if REF in runs:
    ref = runs[REF]
    print(f"\nSo sánh ghép cặp với {REF} (chỉ lần chạy đó đúng / chỉ mốc đúng, p McNemar)")
    print(f"{'lần chạy':<22}{'HE':>24}{'HE+':>24}")
    for name, r in runs.items():
        if name == REF:
            continue
        cells = []
        for i in (0, 1):
            a, b, p = mcnemar({t: v[i] for t, v in r.items()}, {t: ref[t][i] for t in r})
            cells.append(f"{a} / {b}  p={p:.3f}")
        print(f"{name:<22}{cells[0]:>24}{cells[1]:>24}")
else:
    print(f"\n(không có lần chạy mốc '{REF}' nên bỏ qua so sánh ghép cặp)")

with open(f"{RESULTS}/summary.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["run", "n", "HE_pass", "HE_pct", "HE+_pass", "HE+_pct"])
    w.writerows(rows)
print(f"\nĐã ghi {RESULTS}/summary.csv")

lần chạy                          HE           HE+   95% CI của HE+
base_base               102/164 62.2   87/164 53.0   45.4 – 60.5
instruct_chatml         149/164 90.9  138/164 84.1   77.8 – 88.9
v4_base                 123/164 75.0  112/164 68.3   60.8 – 74.9
v4_chatml               141/164 86.0  130/164 79.3   72.4 – 84.8

So sánh ghép cặp với instruct_chatml (chỉ lần chạy đó đúng / chỉ mốc đúng, p McNemar)
lần chạy                                    HE                     HE+
base_base                      2 / 49  p=0.000         4 / 55  p=0.000
v4_base                        4 / 30  p=0.000         6 / 32  p=0.000
v4_chatml                      3 / 11  p=0.057         4 / 12  p=0.077

Đã ghi evalplus_results/he_20261102_0728/summary.csv
